# IndexTTS-2.5 Kaggle 部署

在 Kaggle GPU（P100/T4）上运行 **IndexTTS-2.5**（支持中/英/日/西/阿多语言 + 语音克隆 + 情感控制）。

运行前请确认：
- 右侧 **Settings → Accelerator** 选择 **GPU**（T4 x2 或 P100）。
- **Settings → Internet** 必须**开启**（下载依赖和权重需要联网）。

> 若 HuggingFace 下载慢，可在下方「下载模型」步骤前运行 `%env HF_ENDPOINT=https://hf-mirror.com`。
>
> 所有生成文件保存在 `/kaggle/working/`，提交/运行完成后可在 Output 标签页下载。

In [ ]:
# 1. 检查 GPU
!nvidia-smi

In [ ]:
# 2. 克隆仓库（默认用上游 index-tts/index-tts；如需 fork 改为你的地址）
REPO_URL = "https://github.com/index-tts/index-tts.git"
!git clone --depth 1 $REPO_URL /kaggle/working/index-tts
%cd /kaggle/working/index-tts

In [ ]:
# 3. 安装 uv 并同步依赖（含 webui / deepspeed 等全部 extras）
# Kaggle 已内置较新 Python 3.11，项目要求 >=3.10,<3.12，可直接用系统 Python 装 uv。
!pip install -U uv -q
!uv sync --all-extras
# 国内镜像加速（可选）:
# !uv sync --all-extras --default-index "https://mirrors.aliyun.com/pypi/simple"

In [ ]:
# 4. 下载 IndexTTS-2.5 权重到 checkpoints/（约 3~4 GB，视网速约 5~15 分钟）
!uv tool install huggingface-hub -q
!hf download IndexTeam/IndexTTS-2.5 --local-dir=checkpoints
# 备选：ModelScope
# !uv tool install modelscope -q
# !modelscope download --model IndexTeam/IndexTTS-2.5 --local_dir checkpoints

In [ ]:
# 5. 环境自检
!uv run tools/gpu_check.py

In [ ]:
# 6. 初始化 IndexTTS-2.5（BF16 省显存；示例音频首次运行自动下载）
from indextts.utils.examples_downloader import ensure_examples_available
ensure_examples_available()

from indextts.infer_v2_5 import IndexTTS2
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)

In [ ]:
# 7. 语音克隆（单参考音频 + 指定语言）
tts.infer(
    spk_audio_prompt="examples/voice_01.wav",
    text="你好，我是 IndexTTS-2.5，欢迎测试多语言语音合成。",
    lang="ZH",
    output_path="output_zh.wav",
    verbose=True,
)

In [ ]:
# 8. 情感控制（情感参考音频 + emo_alpha 强度）
tts.infer(
    spk_audio_prompt="examples/voice_07.wav",
    text="酒楼丧尽天良，开始借机竞拍房间，哎，一群蠢货。",
    lang="ZH",
    output_path="output_emo.wav",
    emo_audio_prompt="examples/emo_sad.wav",
    emo_alpha=0.9,
    verbose=True,
)

In [ ]:
# 9. 确认生成结果（后续可在 Output 标签页下载）
!ls -lh /kaggle/working/*.wav

## 说明

- 生成的 WAV 在 `/kaggle/working/` 下（output_zh.wav、output_emo.wav），Notebook 提交运行结束后自动打包为 Output 供下载。
- Kaggle 会话最长 12 小时（GPU），足够完成权重下载 + 推理。
- 若只想快速验证，可跳过第 4 步改用较小的 IndexTTS-2 权重（`IndexTeam/IndexTTS-2` 到 `checkpoints_2`），并在第 6 步改用 `indextts.infer_v2.IndexTTS2`。